[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ofermend/hands-on-rag/blob/main/chapter9/create-graph.ipynb)

# Chapter 9: Chunk-Centric IMDB + MovieSum Knowledge Graph

This notebook creates a **chunk-centric** movie knowledge graph where script chunks are the foundation and all other entities are connected only if they relate to movies with scripts.

**What you'll learn:**
- Load and integrate MovieSum scripts with IMDB metadata
- Chunk movie scripts and extract character entities using regex patterns
- Build a Neo4j knowledge graph with Movies, Persons, Characters, Genres, and Chunks
- Create relationships including ACTED_IN, DIRECTED, APPEARS_IN, MENTIONS, and PORTRAYED_BY

**Prerequisites:** `pip install datasets langchain langchain-text-splitters neo4j` and a running Neo4j instance.

In [1]:
## Step 1: Import Libraries and Configuration

We install the required packages and configure the Neo4j connection, IMDB data sources, and text chunking parameters.

In [2]:
!pip install --quiet datasets langchain langchain-text-splitters neo4j


[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: pip install --upgrade pip


In [ ]:
# Import all required libraries
import subprocess
import sys
import os

import pandas as pd
import gzip
import urllib.request
from pathlib import Path
from typing import List, Dict, Any, Optional
import re
from tqdm import tqdm

# Import ML/Graph libraries
from datasets import load_dataset
from langchain_text_splitters import RecursiveCharacterTextSplitter
from neo4j import GraphDatabase

In [4]:
# Configuration; update this to point to your Neo4J instance
NEO4J_URI = "bolt://localhost:7687"
NEO4J_USER = "neo4j"
NEO4J_PASSWORD = "hands-on-rag-book"

# Data directory
DATA_DIR = Path("./data")
DATA_DIR.mkdir(exist_ok=True)

# IMDB dataset URLs
IMDB_BASE_URL = "https://datasets.imdbws.com/"
IMDB_DATASETS = {
    'title_basics': 'title.basics.tsv.gz',
    'name_basics': 'name.basics.tsv.gz', 
    'title_principals': 'title.principals.tsv.gz',
    'title_crew': 'title.crew.tsv.gz'
}

# Text chunking configuration
CHUNK_SIZE = 1024
CHUNK_OVERLAP = 100

We pull the MovieSum dataset (1,800 movie scripts) from HuggingFace and download IMDB metadata files, filtering them to only the movies that have scripts.

## Step 2: Load Both Datasets

We chunk every movie script with LangChain's text splitter and extract character names using regex patterns, preparing all the raw material for graph construction.

In [5]:
def download_imdb_dataset(dataset_name: str, filename: str) -> Path:
    """Download IMDB dataset if not already present."""
    file_path = DATA_DIR / filename
    
    if file_path.exists():
        print(f"✓ {dataset_name} already exists ({file_path.stat().st_size / (1024*1024):.1f} MB)")
        return file_path
    
    url = IMDB_BASE_URL + filename
    print(f"Downloading {dataset_name}...")
    
    try:
        urllib.request.urlretrieve(url, file_path)
        size_mb = file_path.stat().st_size / (1024 * 1024)
        print(f"✓ Downloaded {dataset_name} ({size_mb:.1f} MB)")
        return file_path
    except Exception as e:
        print(f"✗ Error downloading {dataset_name}: {e}")
        return None

def load_imdb_tsv(file_path: Path, moviesum_ids: set = None) -> pd.DataFrame:
    """Load IMDB TSV file with optional filtering by MovieSum IDs."""
    try:
        if file_path.suffix == '.gz':
            with gzip.open(file_path, 'rt', encoding='utf-8') as f:
                df = pd.read_csv(f, sep='\t', na_values='\\N')
        else:
            df = pd.read_csv(file_path, sep='\t', na_values='\\N')
        
        # Filter by MovieSum IDs if provided and applicable
        if moviesum_ids and 'tconst' in df.columns:
            original_size = len(df)
            df = df[df['tconst'].isin(moviesum_ids)].copy()
            print(f"  Filtered {file_path.name}: {len(df):,} / {original_size:,} records")
        else:
            print(f"  Loaded {file_path.name}: {len(df):,} records")
        
        return df
    except Exception as e:
        print(f"  ✗ Error loading {file_path}: {e}")
        return pd.DataFrame()

# Load MovieSum dataset first to get the movie filter
print("=== LOADING MOVIESUM DATASET ===")
try:
    moviesum_dataset = load_dataset("rohitsaxena/MovieSum")
    moviesum_data = moviesum_dataset['train']
    moviesum_df = pd.DataFrame(moviesum_data)
    
    # Extract IMDB IDs for filtering
    moviesum_imdb_ids = set(moviesum_df['imdb_id'].tolist())
    
    print(f"✓ MovieSum loaded: {len(moviesum_df):,} movies")
    print(f"✓ Filter set: {len(moviesum_imdb_ids):,} unique IMDB IDs")
    print(f"  Sample IDs: {list(moviesum_imdb_ids)[:5]}")
    
except Exception as e:
    print(f"✗ Error loading MovieSum: {e}")
    moviesum_df = pd.DataFrame()
    moviesum_imdb_ids = set()

# Download and load IMDB datasets with immediate filtering
print(f"\n=== LOADING IMDB DATASETS (filtered to {len(moviesum_imdb_ids)} movies) ===")
imdb_dataframes = {}

for dataset_name, filename in IMDB_DATASETS.items():
    print(f"\nProcessing {dataset_name}...")
    file_path = download_imdb_dataset(dataset_name, filename)
    
    if file_path:
        df = load_imdb_tsv(file_path, moviesum_imdb_ids if moviesum_imdb_ids else None)
        imdb_dataframes[dataset_name] = df
    else:
        imdb_dataframes[dataset_name] = pd.DataFrame()

print(f"\n=== LOADING SUMMARY ===")
print(f"MovieSum: {len(moviesum_df):,} movies with scripts")
for name, df in imdb_dataframes.items():
    print(f"{name}: {len(df):,} records")
print("\nBoth datasets loaded and matched")

=== LOADING MOVIESUM DATASET ===


Repo card metadata block was not found. Setting CardData to empty.


✓ MovieSum loaded: 1,800 movies
✓ Filter set: 1,792 unique IMDB IDs
  Sample IDs: ['tt0067185', 'tt8093700', 'tt0115632', 'tt0118750', 'tt1045772']

=== LOADING IMDB DATASETS (filtered to 1792 movies) ===

Processing title_basics...
✓ title_basics already exists (202.4 MB)


/var/folders/y0/qd7p5ft96k7br3ztvfwfbsqc0000gn/T/ipykernel_48792/1625432030.py:26: DtypeWarning: Columns (7) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(f, sep='\t', na_values='\\N')


  Filtered title.basics.tsv.gz: 1,792 / 11,981,307 records

Processing name_basics...
✓ name_basics already exists (278.3 MB)
  Loaded name.basics.tsv.gz: 14,781,125 records

Processing title_principals...
✓ title_principals already exists (701.7 MB)
  Filtered title.principals.tsv.gz: 38,738 / 95,276,239 records

Processing title_crew...
✓ title_crew already exists (74.6 MB)
  Filtered title.crew.tsv.gz: 1,792 / 11,981,307 records

=== LOADING SUMMARY ===
MovieSum: 1,800 movies with scripts
title_basics: 1,792 records
name_basics: 14,781,125 records
title_principals: 38,738 records
title_crew: 1,792 records

Both datasets loaded and matched


We assemble the node and relationship lists — Movies, Persons, Characters, Genres, and Chunks — ensuring every entity connects back to a movie that has a script.

## Step 3: Process All Data

We define helper functions for processing genres, extracting characters from scripts using regex patterns, chunking movie scripts, and then run them all to prepare the data for graph construction.

In [6]:
def process_genres(genres_str: str) -> List[str]:
    """Process genres string into list."""
    if pd.isna(genres_str) or genres_str == '':
        return []
    return [genre.lower().strip() for genre in genres_str.split(',')]

def process_professions(professions_str: str) -> List[str]:
    """Process professions string into list."""
    if pd.isna(professions_str) or professions_str == '':
        return []
    return [prof.lower().strip() for prof in professions_str.split(',')]

def process_known_for_titles(titles_str: str) -> List[str]:
    """Process known for titles string into list."""
    if pd.isna(titles_str) or titles_str == '':
        return []
    return [title.strip() for title in titles_str.split(',')]

def extract_characters_from_script(script_text: str, movie_title: str) -> List[str]:
    """Extract character names from movie script using simple regex patterns."""
    if pd.isna(script_text) or not script_text:
        return []
    
    characters = set()
    
    # Pattern 1: Lines that start with character names (all caps, followed by colon or newline)
    # Example: "JOHN:", "MARY\n"
    char_pattern1 = re.findall(r'^([A-Z][A-Z\s]{2,20}?)(?::|$)', script_text, re.MULTILINE)
    characters.update([name.strip() for name in char_pattern1 if len(name.strip()) > 2])
    
    # Pattern 2: Character names in parentheticals
    # Example: "(JOHN enters)", "(MARY speaking)"
    char_pattern2 = re.findall(r'\(([A-Z][A-Z\s]{2,15}?)(?:\s+[a-z]|\))', script_text)
    characters.update([name.strip() for name in char_pattern2 if len(name.strip()) > 2])
    
    # Pattern 3: Stage directions with character names
    # Example: "John walks to the door", "Mary looks surprised"
    words = script_text.split()
    potential_names = [word for word in words if word.istitle() and len(word) > 2 and word.isalpha()]
    
    # Filter to keep only frequently mentioned names (likely characters)
    name_counts = {}
    for name in potential_names:
        name_counts[name] = name_counts.get(name, 0) + 1
    
    # Keep names mentioned at least 3 times
    frequent_names = [name for name, count in name_counts.items() if count >= 3]
    characters.update(frequent_names)
    
    # Clean and filter characters
    cleaned_chars = set()
    for char in characters:
        char = char.strip().upper()
        # Filter out common words that aren't character names
        if (len(char) >= 3 and len(char) <= 20 and 
            char not in {'THE', 'AND', 'BUT', 'FOR', 'WITH', 'FROM', 'INTO', 'OVER', 'UNDER',
                        'FADE', 'CUT', 'INT', 'EXT', 'DAY', 'NIGHT', 'SCENE', 'ACT', 'END',
                        'VOICE', 'SOUND', 'MUSIC', 'TITLE', 'CREDITS', 'CONT', 'CONTINUED',
                        'MONTAGE', 'FLASHBACK', 'LATER', 'MEANWHILE', 'CLOSE', 'WIDE', 'PAN'}):
            cleaned_chars.add(char.lower())
    
    return list(cleaned_chars)

def chunk_movie_scripts(moviesum_df: pd.DataFrame) -> pd.DataFrame:
    """Chunk all movie scripts using LangChain."""
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=CHUNK_SIZE,
        chunk_overlap=CHUNK_OVERLAP,
        length_function=len,
        is_separator_regex=False,
    )
    
    all_chunks = []
    processed_movies = 0
    
    print(f"Chunking {len(moviesum_df)} movie scripts...")
    
    for _, row in tqdm(moviesum_df.iterrows(), total=len(moviesum_df), desc="Processing scripts"):
        imdb_id = row['imdb_id']
        movie_name = row['movie_name']
        script_content = row['script']
        
        if pd.notna(script_content) and len(script_content.strip()) > 0:
            try:
                # Clean script content (remove XML tags)
                cleaned_script = re.sub(r'<[^>]+>', ' ', script_content)
                cleaned_script = re.sub(r'\s+', ' ', cleaned_script).strip()
                
                if len(cleaned_script) > 100:
                    chunks = text_splitter.split_text(cleaned_script)
                    
                    for chunk_idx, chunk_text in enumerate(chunks):
                        all_chunks.append({
                            'chunk_id': f"{imdb_id}_chunk_{chunk_idx}",
                            'movie_imdb_id': imdb_id,
                            'movie_name': movie_name,
                            'chunk_index': chunk_idx,
                            'text': chunk_text.strip(),
                            'text_length': len(chunk_text)
                        })
                    
                    processed_movies += 1
                        
            except Exception as e:
                print(f"Error processing {movie_name}: {e}")
                continue
    
    chunks_df = pd.DataFrame(all_chunks)
    print(f"✓ Processed {processed_movies} movies into {len(chunks_df):,} chunks")
    print(f"  Average: {len(chunks_df)/processed_movies:.1f} chunks per movie")
    print(f"  Chunk size: {chunks_df['text_length'].mean():.0f} ± {chunks_df['text_length'].std():.0f} chars")
    
    return chunks_df

def extract_script_characters(moviesum_df: pd.DataFrame) -> pd.DataFrame:
    """Extract characters from movie scripts using NLP techniques."""
    all_characters = []
    
    print(f"Extracting characters from {len(moviesum_df)} movie scripts...")
    
    for _, row in tqdm(moviesum_df.iterrows(), total=len(moviesum_df), desc="Extracting characters"):
        imdb_id = row['imdb_id']
        movie_name = row['movie_name']
        script_content = row['script']
        
        if pd.notna(script_content) and len(script_content.strip()) > 0:
            try:
                # Extract characters from script
                characters = extract_characters_from_script(script_content, movie_name)
                
                for char_name in characters:
                    all_characters.append({
                        'character_name': char_name,
                        'movie_imdb_id': imdb_id,
                        'movie_name': movie_name,
                        'source': 'script_extraction'
                    })
                        
            except Exception as e:
                print(f"Error extracting characters from {movie_name}: {e}")
                continue
    
    chars_df = pd.DataFrame(all_characters)
    if not chars_df.empty:
        print(f"✓ Extracted {len(chars_df):,} character mentions from {chars_df['movie_imdb_id'].nunique():,} movies")
        print(f"  Unique characters: {chars_df['character_name'].nunique():,}")
        print(f"  Average: {len(chars_df)/chars_df['movie_imdb_id'].nunique():.1f} characters per movie")
    else:
        print("✗ No characters extracted from scripts")
    
    return chars_df

print("=== PROCESSING ALL DATA ===")

# Process movie scripts into chunks
chunks_df = chunk_movie_scripts(moviesum_df)

# Extract characters from scripts
script_characters_df = extract_script_characters(moviesum_df)

# Process IMDB data
titles_df = imdb_dataframes.get('title_basics', pd.DataFrame())
names_df = imdb_dataframes.get('name_basics', pd.DataFrame())
principals_df = imdb_dataframes.get('title_principals', pd.DataFrame())
crew_df = imdb_dataframes.get('title_crew', pd.DataFrame())

print(f"\n=== DATA PROCESSING SUMMARY ===")
print(f"Movies (IMDB): {len(titles_df):,}")
print(f"Persons (IMDB): {len(names_df):,}")
print(f"Principals (IMDB): {len(principals_df):,}")
print(f"Crew (IMDB): {len(crew_df):,}")
print(f"Movie Scripts: {len(moviesum_df):,}")
print(f"Script Chunks: {len(chunks_df):,}")
print(f"Script Characters: {len(script_characters_df):,}")

=== PROCESSING ALL DATA ===
Chunking 1800 movie scripts...


Processing scripts: 100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 1800/1800 [00:28<00:00, 62.47it/s]


✓ Processed 1800 movies into 253,007 chunks
  Average: 140.6 chunks per movie
  Chunk size: 1017 ± 45 chars
Extracting characters from 1800 movie scripts...


Extracting characters: 100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 1800/1800 [00:04<00:00, 382.80it/s]


✓ Extracted 131,696 character mentions from 1,792 movies
  Unique characters: 22,894
  Average: 73.5 characters per movie

=== DATA PROCESSING SUMMARY ===
Movies (IMDB): 1,792
Persons (IMDB): 14,781,125
Principals (IMDB): 38,738
Crew (IMDB): 1,792
Movie Scripts: 1,800
Script Chunks: 253,007
Script Characters: 131,696


## Step 4: Create Graph Data Structures

We assemble deduplicated node lists (Movies, Persons, Characters, Genres, Chunks) and all relationship tuples, ensuring every entity traces back to a movie that has a script.

In [7]:
def create_all_graph_data(titles_df, names_df, principals_df, crew_df, chunks_df, script_characters_df=None):
    """Create chunk-centric graph data structures - only include entities connected to movies with scripts."""
    print("=== CREATING CHUNK-CENTRIC GRAPH DATA STRUCTURES ===")
    
    # 0. Start with chunks and get the movies that have scripts
    processed_chunks = []
    chunk_movie_relationships = []
    movies_with_scripts = set()
    
    if not chunks_df.empty:
        print(f"Processing {len(chunks_df)} chunks as the graph foundation...")
        
        for _, row in chunks_df.iterrows():
            chunk_data = {
                'chunk_id': row['chunk_id'],
                'text': row['text'],
                'chunk_index': int(row['chunk_index']),
                'text_length': int(row['text_length'])
            }
            processed_chunks.append(chunk_data)
            
            chunk_movie_relationships.append({
                'chunk_id': row['chunk_id'],
                'movie_id': row['movie_imdb_id']
            })
            
            # Track movies that have scripts
            movies_with_scripts.add(row['movie_imdb_id'])
        
        print(f"✓ Found {len(movies_with_scripts)} movies with scripts")
    else:
        print("ERROR: No chunks available - cannot create chunk-centric graph")
        return {}
    
    # 1. Process ONLY movies that have scripts
    processed_movies = []
    all_genres = set()
    movie_genre_relationships = []
    
    if not titles_df.empty:
        # Filter for movies that have scripts AND are in IMDB
        movies_df = titles_df[
            (titles_df['titleType'] == 'movie') & 
            (titles_df['tconst'].isin(movies_with_scripts))
        ].copy()
        
        print(f"Processing {len(movies_df)} movies (filtered to those with scripts)...")
        
        for _, row in movies_df.iterrows():
            genres = process_genres(row['genres'])
            movie_data = {
                'imdb_id': row['tconst'],
                'title': row['primaryTitle'].strip() if pd.notna(row['primaryTitle']) else '',
                'original_title': row['originalTitle'].strip() if pd.notna(row['originalTitle']) else '',
                'start_year': int(row['startYear']) if pd.notna(row['startYear']) and str(row['startYear']).isdigit() else 0,
                'end_year': int(row['endYear']) if pd.notna(row['endYear']) and str(row['endYear']).isdigit() else 0,
                'runtime_minutes': int(row['runtimeMinutes']) if pd.notna(row['runtimeMinutes']) and str(row['runtimeMinutes']).isdigit() else 0,
                'is_adult': bool(row['isAdult']) if pd.notna(row['isAdult']) else False,
                'genres': genres
            }
            processed_movies.append(movie_data)
            all_genres.update(genres)
            
            # Create movie-genre relationships
            for genre_name in genres:
                movie_genre_relationships.append({
                    'movie_id': movie_data['imdb_id'],
                    'genre_name': genre_name
                })
    
    processed_genres = [{'name': genre} for genre in sorted(all_genres) if genre]
    
    # 2. Process characters from scripts (primary source)
    processed_characters = []
    character_movie_relationships = []
    chunk_character_relationships = []  # NEW: Direct chunk-character links
    all_characters = set()
    
    if script_characters_df is not None and not script_characters_df.empty:
        print(f"Processing {len(script_characters_df)} script-extracted characters...")
        
        # Filter characters to only those from movies with scripts in our graph
        valid_movie_ids = {movie['imdb_id'] for movie in processed_movies}
        filtered_chars = script_characters_df[
            script_characters_df['movie_imdb_id'].isin(valid_movie_ids)
        ].copy()
        
        for _, row in filtered_chars.iterrows():
            char_name = row['character_name'].strip().lower()
            if char_name and len(char_name) > 1:
                all_characters.add(char_name)
                
                # Create character-movie relationship
                character_movie_relationships.append({
                    'character_name': char_name,
                    'movie_id': row['movie_imdb_id']
                })
        
        print(f"✓ Found {len(all_characters)} unique characters from scripts")
        
        # NEW: Create chunk-character relationships for granular linking
        print(f"Creating chunk-character relationships for granular script analysis...")
        
        # Create a lookup for efficient character searching
        character_movie_lookup = {}
        for _, row in filtered_chars.iterrows():
            movie_id = row['movie_imdb_id']
            char_name = row['character_name'].strip().lower()
            if movie_id not in character_movie_lookup:
                character_movie_lookup[movie_id] = []
            character_movie_lookup[movie_id].append(char_name)
        
        chunk_char_count = 0
        for _, chunk_row in chunks_df.iterrows():
            movie_id = chunk_row['movie_imdb_id']
            chunk_text = chunk_row['text'].lower()
            
            # Only process chunks for movies we have in our graph
            if movie_id in character_movie_lookup:
                # Check which characters appear in this chunk
                for char_name in character_movie_lookup[movie_id]:
                    # Use word boundaries to avoid partial matches
                    import re
                    pattern = r'\b' + re.escape(char_name) + r'\b'
                    if re.search(pattern, chunk_text):
                        chunk_character_relationships.append({
                            'chunk_id': chunk_row['chunk_id'],
                            'character_name': char_name
                        })
                        chunk_char_count += 1
        
        print(f"✓ Created {chunk_char_count:,} chunk-character relationships")
    
    # Create character entities from script extraction
    processed_characters = [{'name': char, 'source': 'script'} for char in sorted(all_characters) if char]
    
    # 3. Process persons - handle corrupted names_df gracefully
    processed_persons = []
    processed_relationships = []
    valid_movie_ids = {movie['imdb_id'] for movie in processed_movies}
    
    if names_df.empty:
        print("⚠ WARNING: No person names data available (names_df is empty)")
        print("  This is likely due to corrupted name.basics.tsv.gz file")
        print("  Proceeding with movies, genres, characters, and chunks only")
        
        # Still process crew data for director relationships if available
        if not crew_df.empty:
            crew_with_directors = crew_df[
                (pd.notna(crew_df['directors'])) &
                (crew_df['tconst'].isin(valid_movie_ids))
            ]
            
            print(f"Processing {len(crew_with_directors)} directing relationships without person details...")
            
            # Create director relationships with director IDs only (no person details)
            for _, row in crew_with_directors.iterrows():
                directors = str(row['directors']).split(',')
                for director_id in directors:
                    director_id = director_id.strip()
                    if director_id:
                        # Create basic person record with just ID
                        person_data = {
                            'imdb_id': director_id,
                            'name': f'director_{director_id}',  # Placeholder name
                            'birth_year': 0,
                            'death_year': 0,
                            'professions': ['director'],
                            'known_for_titles': []
                        }
                        processed_persons.append(person_data)
                        
                        processed_relationships.append({
                            'person_id': director_id,
                            'movie_id': row['tconst'],
                            'relationship_type': 'DIRECTED',
                            'character': '',
                            'ordering': 0
                        })
            
            # Remove duplicates from processed_persons
            seen_ids = set()
            unique_persons = []
            for person in processed_persons:
                if person['imdb_id'] not in seen_ids:
                    unique_persons.append(person)
                    seen_ids.add(person['imdb_id'])
            processed_persons = unique_persons
    
    else:
        # Original logic for when names_df is available
        if not principals_df.empty and not crew_df.empty:
            # Find person IDs connected to our movies
            connected_person_ids = set()
            
            # Get actors from principals
            if not principals_df.empty:
                actor_principals = principals_df[
                    (principals_df['category'].isin(['actor', 'actress'])) &
                    (principals_df['tconst'].isin(valid_movie_ids))
                ]
                connected_person_ids.update(actor_principals['nconst'].tolist())
            
            # Get directors from crew
            if not crew_df.empty:
                crew_with_directors = crew_df[
                    (pd.notna(crew_df['directors'])) &
                    (crew_df['tconst'].isin(valid_movie_ids))
                ]
                for _, row in crew_with_directors.iterrows():
                    directors = str(row['directors']).split(',')
                    connected_person_ids.update([d.strip() for d in directors if d.strip()])
            
            print(f"Found {len(connected_person_ids)} persons connected to movies with scripts")
            
            # Filter persons to only those connected to our movies
            connected_persons_df = names_df[
                (names_df['nconst'].isin(connected_person_ids)) &
                (names_df['primaryProfession'].str.contains('actor|actress|director', case=False, na=False))
            ].copy()
            
            print(f"Processing {len(connected_persons_df)} connected persons...")
            
            for _, row in connected_persons_df.iterrows():
                person_data = {
                    'imdb_id': row['nconst'],
                    'name': row['primaryName'].strip().lower() if pd.notna(row['primaryName']) else '',
                    'birth_year': int(row['birthYear']) if pd.notna(row['birthYear']) and str(row['birthYear']).isdigit() else 0,
                    'death_year': int(row['deathYear']) if pd.notna(row['deathYear']) and str(row['deathYear']).isdigit() else 0,
                    'professions': process_professions(row['primaryProfession']),
                    'known_for_titles': process_known_for_titles(row['knownForTitles'])
                }
                processed_persons.append(person_data)
    
    # Create sets for efficient lookup (only entities connected to movies with scripts)
    valid_persons = {person['imdb_id'] for person in processed_persons}

    print(f"Processing relationships for {len(valid_movie_ids)} movies and {len(valid_persons)} persons...")

    # Process principals (actors) - only if we have both names and principals data
    if not names_df.empty and not principals_df.empty:
        actor_principals = principals_df[
            (principals_df['category'].isin(['actor', 'actress'])) &
            (principals_df['tconst'].isin(valid_movie_ids)) &
            (principals_df['nconst'].isin(valid_persons))
        ].copy()

        print(f"Processing {len(actor_principals)} acting relationships...")

        for _, row in actor_principals.iterrows():
            characters = row['characters'] if pd.notna(row['characters']) else ''
            
            # Also create direct person-movie relationship (ACTED_IN)
            processed_relationships.append({
                'person_id': row['nconst'],
                'movie_id': row['tconst'],
                'relationship_type': 'ACTED_IN',
                'character': characters,
                'ordering': int(row['ordering']) if pd.notna(row['ordering']) else 999
            })

    # Process crew (directors) - only if we have the data and persons
    if not crew_df.empty and processed_persons:
        crew_with_directors = crew_df[
            (pd.notna(crew_df['directors'])) &
            (crew_df['tconst'].isin(valid_movie_ids))
        ].copy()

        print(f"Processing {len(crew_with_directors)} directing relationships...")

        for _, row in crew_with_directors.iterrows():
            directors = str(row['directors']).split(',')
            for director_id in directors:
                director_id = director_id.strip()
                if director_id and director_id in valid_persons:
                    processed_relationships.append({
                        'person_id': director_id,
                        'movie_id': row['tconst'],
                        'relationship_type': 'DIRECTED',
                        'character': '',
                        'ordering': 0
                    })
    
    # NEW: Create PORTRAYED_BY relationships by matching ACTED_IN character data with Character nodes
    portrayed_by_relationships = []
    if all_characters and processed_relationships:
        print(f"Creating PORTRAYED_BY relationships by matching character names...")
        
        # Get ACTED_IN relationships with character info
        acted_in_with_chars = [r for r in processed_relationships 
                              if r['relationship_type'] == 'ACTED_IN' and r.get('character')]
        
        portrayed_count = 0
        for rel in acted_in_with_chars:
            character_info = rel['character']
            if not character_info:
                continue
                
            # Clean and extract character names from IMDB character field
            # Character field often contains things like: '["Alec Trevelyan"]', 'James Bond', etc.
            character_names = []
            
            # Remove brackets and quotes, split by commas
            cleaned_chars = re.sub(r'[\[\]"\'()]', '', character_info)
            raw_names = [name.strip() for name in cleaned_chars.split(',')]
            
            for raw_name in raw_names:
                if raw_name and len(raw_name) > 1:
                    # Convert to lowercase for matching
                    clean_name = raw_name.lower().strip()
                    character_names.append(clean_name)
            
            # Try to match with script-extracted characters
            for char_name in character_names:
                if char_name in all_characters:
                    portrayed_by_relationships.append({
                        'character_name': char_name,
                        'person_id': rel['person_id'],
                        'ordering': rel['ordering']
                    })
                    portrayed_count += 1
                else:
                    # Try fuzzy matching - check if any character contains this name or vice versa
                    for script_char in all_characters:
                        # Check if script character name is contained in IMDB character name
                        if (len(script_char) >= 3 and script_char in char_name) or \
                           (len(char_name) >= 3 and char_name in script_char):
                            portrayed_by_relationships.append({
                                'character_name': script_char,
                                'person_id': rel['person_id'],
                                'ordering': rel['ordering']
                            })
                            portrayed_count += 1
                            break
        
        print(f"✓ Created {portrayed_count:,} PORTRAYED_BY relationships")
    
    # Show relationship breakdown
    rel_types = {}
    for rel in processed_relationships:
        rel_type = rel['relationship_type']
        rel_types[rel_type] = rel_types.get(rel_type, 0) + 1
    
    print(f"\n=== CHUNK-CENTRIC GRAPH DATA SUMMARY ===")
    print(f"Foundation: {len(processed_chunks):,} script chunks")
    print(f"Movies (with scripts): {len(processed_movies):,}")
    print(f"Genres (from script movies): {len(processed_genres):,}")
    print(f"Persons (connected to script movies): {len(processed_persons):,}")
    print(f"Characters (from script extraction): {len(processed_characters):,}")
    print(f"Chunk-Movie relationships: {len(chunk_movie_relationships):,}")
    print(f"Movie-Genre relationships: {len(movie_genre_relationships):,}")
    print(f"Character-Movie relationships: {len(character_movie_relationships):,}")
    print(f"Chunk-Character relationships: {len(chunk_character_relationships):,}")  # NEW
    print(f"PORTRAYED_BY relationships: {len(portrayed_by_relationships):,}")  # NEW
    print(f"Person-Movie/Character relationships:")
    for rel_type, count in rel_types.items():
        print(f"  {rel_type}: {count:,}")
    
    # Verify everything is connected to chunks
    movies_with_chunks = {rel['movie_id'] for rel in chunk_movie_relationships}
    movies_in_graph = {movie['imdb_id'] for movie in processed_movies}
    
    print(f"\n=== CONNECTIVITY VERIFICATION ===")
    print(f"Movies with chunks: {len(movies_with_chunks):,}")
    print(f"Movies in graph: {len(movies_in_graph):,}")
    
    if movies_with_chunks != movies_in_graph:
        missing_movies = movies_in_graph - movies_with_chunks
        extra_chunks = movies_with_chunks - movies_in_graph
        if missing_movies:
            print(f"WARNING: {len(missing_movies)} movies without chunks: {list(missing_movies)[:5]}")
        if extra_chunks:
            print(f"WARNING: {len(extra_chunks)} chunk movies not in graph: {list(extra_chunks)[:5]}")
    
    return {
        'movies': processed_movies,
        'genres': processed_genres,
        'persons': processed_persons,
        'chunks': processed_chunks,
        'characters': processed_characters,
        'relationships': processed_relationships,
        'movie_genre_rels': movie_genre_relationships,
        'chunk_movie_rels': chunk_movie_relationships,
        'character_movie_rels': character_movie_relationships,
        'chunk_character_rels': chunk_character_relationships,  # NEW
        'portrayed_by_rels': portrayed_by_relationships  # NEW
    }

# Create all graph data with script characters
graph_data = create_all_graph_data(titles_df, names_df, principals_df, crew_df, chunks_df, script_characters_df)

=== CREATING CHUNK-CENTRIC GRAPH DATA STRUCTURES ===
Processing 253007 chunks as the graph foundation...
✓ Found 1792 movies with scripts
Processing 1779 movies (filtered to those with scripts)...
Processing 131696 script-extracted characters...
✓ Found 22845 unique characters from scripts
Creating chunk-character relationships for granular script analysis...
✓ Created 3,944,467 chunk-character relationships
Found 11436 persons connected to movies with scripts
Processing 11372 connected persons...
Processing relationships for 1779 movies and 11372 persons...
Processing 18129 acting relationships...
Processing 1779 directing relationships...
Creating PORTRAYED_BY relationships by matching character names...
✓ Created 17,972 PORTRAYED_BY relationships

=== CHUNK-CENTRIC GRAPH DATA SUMMARY ===
Foundation: 253,007 script chunks
Movies (with scripts): 1,779
Genres (from script movies): 21
Persons (connected to script movies): 11,372
Characters (from script extraction): 22,845
Chunk-Movie re

## Step 5: Build the Graph

We define the `GraphBuilder` class that connects to Neo4j, creates uniqueness constraints for each node type, and batch-inserts all nodes and relationships to populate the knowledge graph.

In [8]:
class GraphBuilder:
    def __init__(self, uri, user, password):
        self.driver = GraphDatabase.driver(uri, auth=(user, password))
        print("Connected to Neo4j")
    
    def close(self):
        self.driver.close()
        print("Neo4j connection closed")
    
    def clear_database(self):
        """Clear existing data and constraints."""
        with self.driver.session() as session:
            session.run("MATCH (n) DETACH DELETE n")
            print("Database cleared")
            
            # Drop existing constraints
            try:
                result = session.run("SHOW CONSTRAINTS")
                constraints = [record["name"] for record in result if "name" in record]
                
                for constraint_name in constraints:
                    try:
                        session.run(f"DROP CONSTRAINT {constraint_name}")
                        print(f"Dropped constraint: {constraint_name}")
                    except Exception as e:
                        print(f"Could not drop constraint {constraint_name}: {e}")
                        
                if constraints:
                    print(f"Dropped {len(constraints)} existing constraints")
                else:
                    print("No existing constraints to drop")
                    
            except Exception as e:
                print(f"Note: Could not check/drop constraints: {e}")
    
    def create_constraints(self):
        """Create all unique constraints."""
        constraints = [
            "CREATE CONSTRAINT movie_imdb_id IF NOT EXISTS FOR (m:Movie) REQUIRE m.imdb_id IS UNIQUE",
            "CREATE CONSTRAINT person_imdb_id IF NOT EXISTS FOR (p:Person) REQUIRE p.imdb_id IS UNIQUE",
            "CREATE CONSTRAINT genre_name IF NOT EXISTS FOR (g:Genre) REQUIRE g.name IS UNIQUE",
            "CREATE CONSTRAINT character_name IF NOT EXISTS FOR (c:Character) REQUIRE c.name IS UNIQUE",
            "CREATE CONSTRAINT chunk_id IF NOT EXISTS FOR (ch:Chunk) REQUIRE ch.chunk_id IS UNIQUE"
        ]
        
        with self.driver.session() as session:
            for constraint in constraints:
                try:
                    session.run(constraint)
                    constraint_name = constraint.split()[2]
                    print(f"✓ Created constraint: {constraint_name}")
                except Exception as e:
                    print(f"Constraint note: {e}")
        print("All constraints created successfully")
    
    def create_nodes_batch(self, node_type: str, nodes: List[Dict], batch_size: int = 1000):
        """Generic batch node creation."""
        if not nodes:
            print(f"No {node_type} nodes to create")
            return
            
        print(f"Creating {len(nodes):,} {node_type} nodes...")
        
        # Define queries for different node types
        queries = {
            'Movie': """
                UNWIND $items AS item
                MERGE (m:Movie {imdb_id: item.imdb_id})
                SET m.title = item.title,
                    m.original_title = item.original_title,
                    m.start_year = item.start_year,
                    m.end_year = item.end_year,
                    m.runtime_minutes = item.runtime_minutes,
                    m.is_adult = item.is_adult,
                    m.genres = item.genres
            """,
            'Person': """
                UNWIND $items AS item
                MERGE (p:Person {imdb_id: item.imdb_id})
                SET p.name = item.name,
                    p.birth_year = item.birth_year,
                    p.death_year = item.death_year,
                    p.professions = item.professions,
                    p.known_for_titles = item.known_for_titles
            """,
            'Genre': """
                UNWIND $items AS item
                MERGE (g:Genre {name: item.name})
            """,
            'Character': """
                UNWIND $items AS item
                MERGE (c:Character {name: item.name})
                SET c.source = COALESCE(item.source, 'unknown')
            """,
            'Chunk': """
                UNWIND $items AS item
                MERGE (ch:Chunk {chunk_id: item.chunk_id})
                SET ch.text = item.text,
                    ch.chunk_index = item.chunk_index,
                    ch.text_length = item.text_length
            """
        }
        
        query = queries.get(node_type)
        if not query:
            print(f"No query defined for node type: {node_type}")
            return
        
        for i in tqdm(range(0, len(nodes), batch_size), desc=f"Creating {node_type} nodes"):
            batch = nodes[i:i+batch_size]
            with self.driver.session() as session:
                session.run(query, items=batch)
        
        print(f"✓ Created {len(nodes):,} {node_type} nodes")
    
    def create_relationships_batch(self, rel_type: str, relationships: List[Dict], batch_size: int = 1000):
        """Generic batch relationship creation."""
        if not relationships:
            print(f"No {rel_type} relationships to create")
            return
            
        print(f"Creating {len(relationships):,} {rel_type} relationships...")
        
        # Define queries for different relationship types
        queries = {
            'HAS_GENRE': """
                UNWIND $items AS item
                MATCH (m:Movie {imdb_id: item.movie_id})
                MATCH (g:Genre {name: item.genre_name})
                MERGE (m)-[r:HAS_GENRE]->(g)
            """,
            'BELONGS_TO': """
                UNWIND $items AS item
                MATCH (ch:Chunk {chunk_id: item.chunk_id})
                MATCH (m:Movie {imdb_id: item.movie_id})
                MERGE (ch)-[r:BELONGS_TO]->(m)
            """,
            'ACTED_IN': """
                UNWIND $items AS item
                MATCH (p:Person {imdb_id: item.person_id})
                MATCH (m:Movie {imdb_id: item.movie_id})
                MERGE (p)-[r:ACTED_IN]->(m)
                SET r.character = item.character,
                    r.ordering = item.ordering
            """,
            'DIRECTED': """
                UNWIND $items AS item
                MATCH (p:Person {imdb_id: item.person_id})
                MATCH (m:Movie {imdb_id: item.movie_id})
                MERGE (p)-[r:DIRECTED]->(m)
            """,
            'PLAYED': """
                UNWIND $items AS item
                MATCH (p:Person {imdb_id: item.person_id})
                MATCH (c:Character {name: item.character_name})
                MERGE (p)-[r:PLAYED]->(c)
                SET r.ordering = item.ordering
            """,
            'APPEARS_IN': """
                UNWIND $items AS item
                MATCH (c:Character {name: item.character_name})
                MATCH (m:Movie {imdb_id: item.movie_id})
                MERGE (c)-[r:APPEARS_IN]->(m)
            """,
            'PORTRAYED_BY': """
                UNWIND $items AS item
                MATCH (c:Character {name: item.character_name})
                MATCH (p:Person {imdb_id: item.person_id})
                MERGE (c)-[r:PORTRAYED_BY]->(p)
                SET r.ordering = item.ordering
            """,
            'MENTIONS': """
                UNWIND $items AS item
                MATCH (ch:Chunk {chunk_id: item.chunk_id})
                MATCH (c:Character {name: item.character_name})
                MERGE (ch)-[r:MENTIONS]->(c)
            """
        }
        
        query = queries.get(rel_type)
        if not query:
            print(f"No query defined for relationship type: {rel_type}")
            return
        
        for i in tqdm(range(0, len(relationships), batch_size), desc=f"Creating {rel_type} relationships"):
            batch = relationships[i:i+batch_size]
            with self.driver.session() as session:
                session.run(query, items=batch)
        
        print(f"✓ Created {len(relationships):,} {rel_type} relationships")
    
    def create_imdb_relationships(self, relationships: List[Dict]):
        """Create IMDB relationships (ACTED_IN, DIRECTED, PLAYED) and derived relationships."""
        if not relationships:
            print("No IMDB relationships to create")
            return
            
        # Group relationships by type
        acted_in_rels = [r for r in relationships if r['relationship_type'] == 'ACTED_IN']
        directed_rels = [r for r in relationships if r['relationship_type'] == 'DIRECTED']
        played_rels = [r for r in relationships if r['relationship_type'] == 'PLAYED']
        
        # Create each relationship type
        self.create_relationships_batch('ACTED_IN', acted_in_rels)
        self.create_relationships_batch('DIRECTED', directed_rels)
        self.create_relationships_batch('PLAYED', played_rels)
        
        # Create reverse PORTRAYED_BY relationships from PLAYED
        if played_rels:
            portrayed_by_rels = [{
                'character_name': rel['character_name'],
                'person_id': rel['person_id'],
                'ordering': rel['ordering']
            } for rel in played_rels]
            self.create_relationships_batch('PORTRAYED_BY', portrayed_by_rels)
    
    def get_statistics(self) -> Dict:
        """Get comprehensive graph statistics."""
        with self.driver.session() as session:
            stats = {}
            
            # Count nodes
            node_types = ['Movie', 'Person', 'Genre', 'Character', 'Chunk']
            for node_type in node_types:
                result = session.run(f"MATCH (n:{node_type}) RETURN count(n) as count")
                stats[node_type] = result.single()['count']
            
            # Count relationships
            result = session.run("""
            MATCH ()-[r]->() 
            RETURN type(r) as rel_type, count(r) as count 
            ORDER BY count DESC
            """)
            
            for record in result:
                stats[record['rel_type']] = record['count']
            
            return stats

In [9]:
builder = GraphBuilder(NEO4J_URI, NEO4J_USER, NEO4J_PASSWORD)

# 1. Clear and setup
builder.clear_database()
builder.create_constraints()

# 2. Create all node types
print("\n--- Creating Nodes ---")
builder.create_nodes_batch('Movie', graph_data['movies'])
builder.create_nodes_batch('Genre', graph_data['genres'])
builder.create_nodes_batch('Person', graph_data['persons'])
builder.create_nodes_batch('Character', graph_data['characters'])
builder.create_nodes_batch('Chunk', graph_data['chunks'])

# 3. Create all relationships
print("\n--- Creating Relationships ---")
# Basic relationships
builder.create_relationships_batch('HAS_GENRE', graph_data['movie_genre_rels'])
builder.create_relationships_batch('BELONGS_TO', graph_data['chunk_movie_rels'])
builder.create_relationships_batch('APPEARS_IN', graph_data['character_movie_rels'])

# IMDB relationships (ACTED_IN, DIRECTED, PLAYED, PORTRAYED_BY)
builder.create_imdb_relationships(graph_data['relationships'])

# NEW: Character-Person relationships (PORTRAYED_BY)
builder.create_relationships_batch('PORTRAYED_BY', graph_data['portrayed_by_rels'])

# NEW: Chunk-Character relationships (MENTIONS)
builder.create_relationships_batch('MENTIONS', graph_data['chunk_character_rels'])

# 4. Get final statistics
stats = builder.get_statistics()

print("\n=== COMPLETE ENHANCED MOVIE KNOWLEDGE GRAPH ===")
print("\nNodes:")
for node_type in ['Movie', 'Genre', 'Person', 'Character', 'Chunk']:
    count = stats.get(node_type, 0)
    if count > 0:
        print(f"  {node_type}: {count:,}")

print("\nRelationships:")
relationship_types = [key for key in stats.keys() 
                     if key not in ['Movie', 'Genre', 'Person', 'Character', 'Chunk']]

for rel_type in sorted(relationship_types):
    print(f"  {rel_type}: {stats[rel_type]:,}")    

Connected to Neo4j
Database cleared
Note: Could not check/drop constraints: {code: Neo.TransientError.General.MemoryPoolOutOfMemoryError} {message: The allocation of an extra 2.0 MiB would use more than the limit 6.3 GiB. Currently using 6.3 GiB. dbms.memory.transaction.total.max threshold reached}
✓ Created constraint: movie_imdb_id
✓ Created constraint: person_imdb_id
✓ Created constraint: genre_name
✓ Created constraint: character_name
✓ Created constraint: chunk_id
All constraints created successfully

--- Creating Nodes ---
Creating 1,779 Movie nodes...


Creating Movie nodes: 100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 2/2 [00:00<00:00,  3.47it/s]


✓ Created 1,779 Movie nodes
Creating 21 Genre nodes...


Creating Genre nodes: 100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 1/1 [00:00<00:00, 33.48it/s]


✓ Created 21 Genre nodes
Creating 11,372 Person nodes...


...

✓ Created 1,846 DIRECTED relationships
No PLAYED relationships to create
Creating 17,972 PORTRAYED_BY relationships...


Creating PORTRAYED_BY relationships: 100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 18/18 [00:00<00:00, 24.51it/s]


✓ Created 17,972 PORTRAYED_BY relationships
Creating 3,944,467 MENTIONS relationships...


Creating MENTIONS relationships: 100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 3945/3945 [00:57<00:00, 68.88it/s]


✓ Created 3,944,467 MENTIONS relationships

=== COMPLETE ENHANCED MOVIE KNOWLEDGE GRAPH ===

Nodes:
  Movie: 1,779
  Genre: 21
  Person: 11,373
  Character: 22,845
  Chunk: 251,851

Relationships:
  ACTED_IN: 17,719
  APPEARS_IN: 130,469
  BELONGS_TO: 250,393
  DIRECTED: 1,848
  HAS_GENRE: 4,582
  MENTIONS: 3,897,056
  PORTRAYED_BY: 25,643
